# 01 - EDA e Ingeniería de Datos (PySpark)

**Proyecto:** Analítica de Cadena de Suministro - Starbucks Costa Rica

Este notebook realiza la **ingeniería de datos** del proyecto usando Apache Spark:

1. Inicializa una `SparkSession` en modo local.
2. Carga los CSV limpios generados por `extract_data.py`.
3. Realiza un análisis exploratorio (esquemas, conteos, nulos).
4. Transforma el histórico de formato **ancho** (wk1..wk174, 174 semanas continuas) a formato **largo** `[SKU, semana, demanda]` con `stack()`.
5. Une dimensiones de producto (costo, vida útil) y marca SKUs de campaña.
6. Persiste los resultados en **Parquet** para reutilizarlos en los modelos.

> Los CSV de entrada se producen ejecutando `python extract_data.py` desde la raíz del proyecto.


In [1]:
import os
import glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# --- Configuracion de entorno para Spark en Windows ---------------------------
# 1) JAVA_HOME: Spark requiere un JDK (Java 17 para Spark 4.x).
if not os.environ.get("JAVA_HOME"):
    candidatos = (
        glob.glob(r"C:\Program Files\Microsoft\jdk-17*")
        + glob.glob(r"C:\Program Files\Eclipse Adoptium\jdk-17*")
        + glob.glob(os.path.expanduser(r"~\AppData\Local\Microsoft\jdk-17*"))
    )
    if candidatos:
        os.environ["JAVA_HOME"] = candidatos[0]
        os.environ["PATH"] = os.path.join(candidatos[0], "bin") + os.pathsep + os.environ.get("PATH", "")

# 2) HADOOP_HOME: en Windows, escribir archivos (Parquet) requiere winutils.exe
#    y hadoop.dll, ubicados en la carpeta hadoop/bin del proyecto.
_root = "." if os.path.isdir("hadoop") else ".."
_hadoop_home = os.path.abspath(os.path.join(_root, "hadoop"))
if os.path.isdir(os.path.join(_hadoop_home, "bin")):
    os.environ["HADOOP_HOME"] = _hadoop_home
    os.environ["PATH"] = os.path.join(_hadoop_home, "bin") + os.pathsep + os.environ.get("PATH", "")

print("JAVA_HOME  :", os.environ.get("JAVA_HOME"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

# SparkSession en modo local usando todos los nucleos disponibles.
spark = (
    SparkSession.builder
    .appName("Starbucks-EDA-DataEngineering")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


JAVA_HOME  : C:\Program Files\Microsoft\jdk-17.0.19.10-hotspot
HADOOP_HOME: D:\Documents\bigdataproject\hadoop


Spark version: 4.1.2


In [2]:
# Localizamos la raiz del proyecto: el notebook puede ejecutarse desde la
# carpeta raiz o desde la subcarpeta notebooks/.
ROOT = "." if os.path.isdir("data") else ".."
DATA_DIR = os.path.join(ROOT, "data")
PARQUET_DIR = os.path.join(DATA_DIR, "parquet")
os.makedirs(PARQUET_DIR, exist_ok=True)

# Cargamos los CSV con inferencia de esquema y encabezado.
df_hist = spark.read.csv(os.path.join(DATA_DIR, "historico_semanal.csv"), header=True, inferSchema=True)
df_inf = spark.read.csv(os.path.join(DATA_DIR, "inf.csv"), header=True, inferSchema=True)
df_camp = spark.read.csv(os.path.join(DATA_DIR, "refuerzo_campanas.csv"), header=True, inferSchema=True)
df_bio = spark.read.csv(os.path.join(DATA_DIR, "bio.csv"), header=True, inferSchema=True)

print("historico_semanal:", df_hist.count(), "filas")
print("inf            :", df_inf.count(), "filas")
print("refuerzo_campanas:", df_camp.count(), "filas")
print("bio            :", df_bio.count(), "filas")

historico_semanal: 436 filas
inf            : 471 filas


refuerzo_campanas: 51 filas
bio            : 33 filas


## Exploración de esquemas y muestra

In [3]:
# Esquema del histórico (snapshot de inventario + 52 semanas de demanda).
df_hist.select("SKU", "DESCRIPCION", "PROVEEDOR", "PROM_AJUST", "EXIST_W11", "TRANSITO", "wk1", "wk2", "wk52").show(5, truncate=40)

+-------+----------------------------------------+------------------------------+------------------+---------+--------+---+---+----+
|    SKU|                             DESCRIPCION|                     PROVEEDOR|        PROM_AJUST|EXIST_W11|TRANSITO|wk1|wk2|wk52|
+-------+----------------------------------------+------------------------------+------------------+---------+--------+---+---+----+
| 100396|"BROCHURE HOLDER 3.75X8.5""  (UNIDADES)"|STARBUCKS CORPORATION Utileria|               0.0|      5.0|     0.0|0.0|0.0| 0.0|
|1007648|                   GAVETAS PARA MASTRENA|STARBUCKS CORPORATION Utileria|               0.0|      2.0|     0.0|0.0|0.0| 0.0|
| 101091|           TINTA PARA PISTOLA DE PRECIOS|STARBUCKS CORPORATION Utileria|               0.0|      0.0|     0.0|0.0|0.0| 0.0|
| 103242|                               CUBO PLEX|STARBUCKS CORPORATION Utileria|1.6500000000000001|     25.0|     9.0|2.0|0.0| 0.0|
|  10508|                      PISTOLA DE PRECIOS|STARBUCKS CORPORATI

In [4]:
# Conteo de nulos en las columnas clave del histórico.
key_cols = ["SKU", "PROVEEDOR", "PROM_AJUST", "EXIST_W11", "TRANSITO"]
df_hist.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in key_cols
]).show()

+---+---------+----------+---------+--------+
|SKU|PROVEEDOR|PROM_AJUST|EXIST_W11|TRANSITO|
+---+---------+----------+---------+--------+
|  0|       16|         0|        0|       0|
+---+---------+----------+---------+--------+



## Paso 1 - Transformación de formato ancho a largo (data melting)

El histórico tiene **174 semanas continuas** como columnas (`wk1..wk174`). Para alimentar modelos de ML necesitamos un formato largo `[SKU, semana, demanda]`. Lo logramos con la función `stack()` de Spark SQL.


In [ ]:
# Detectamos dinamicamente cuantas semanas trae el historico (wk1..wkN).
# La serie real es continua: 2023 (52) + 2024 (52) + 2025 (52) + 2026 parcial = 174 semanas.
import re as _re
semanas_cols = sorted(
    [c for c in df_hist.columns if _re.fullmatch(r"wk\d+", c)],
    key=lambda c: int(c[2:]),
)
N_SEMANAS = len(semanas_cols)
print(f"Semanas historicas detectadas: {N_SEMANAS}")

# Construimos la expresion stack. Casteamos cada semana a double porque Spark
# infiere algunas columnas como INT y otras como DOUBLE, y stack() exige que
# todos los valores apilados compartan el mismo tipo.
stack_parts = ", ".join([f"{i}, cast({col} as double)" for i, col in enumerate(semanas_cols, start=1)])
stack_expr = f"stack({N_SEMANAS}, {stack_parts}) as (semana, demanda)"

df_largo = (
    df_hist
    .select("SKU", "DESCRIPCION", "PROVEEDOR", F.expr(stack_expr))
    .withColumn("semana", F.col("semana").cast(IntegerType()))
    .withColumn("demanda", F.col("demanda").cast("double"))
)

print("Filas en formato largo:", df_largo.count())
df_largo.show(8)


Semanas historicas detectadas: 174


Filas en formato largo: 75864


+------+--------------------+--------------------+------+-------+
|   SKU|         DESCRIPCION|           PROVEEDOR|semana|demanda|
+------+--------------------+--------------------+------+-------+
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     1|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     2|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     3|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     4|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     5|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     6|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     7|    0.0|
|100396|"BROCHURE HOLDER ...|STARBUCKS CORPORA...|     8|    0.0|
+------+--------------------+--------------------+------+-------+
only showing top 8 rows


In [6]:
# Paso 2 - Imputación de nulos: una semana sin registro = 0 unidades vendidas.
df_largo = df_largo.fillna({"demanda": 0.0})
df_largo = df_largo.filter(F.col("SKU").isNotNull())
print("Nulos restantes en demanda:", df_largo.filter(F.col("demanda").isNull()).count())

Nulos restantes en demanda: 0


## Paso 3 - Enriquecimiento: costo del producto y bandera de campaña

In [7]:
# Normalizamos el SKU a texto en todas las tablas para garantizar joins correctos.
for d in ["df_largo", "df_inf", "df_camp"]:
    pass
df_largo = df_largo.withColumn("SKU", F.col("SKU").cast("string"))
df_inf = df_inf.withColumn("SKU", F.col("SKU").cast("string"))
df_camp = df_camp.withColumn("SKU", F.col("SKU").cast("string"))

# Costo unitario por SKU desde la hoja Inf.
inf_costo = df_inf.select("SKU", "COSTO_UNITARIO", "VIDA_UTIL", "CS_POR_PALLET")

# Conjunto de SKUs reforzados por campañas estacionales.
skus_campana = df_camp.select("SKU").distinct().withColumn("es_campana", F.lit(1))

df_modelo = (
    df_largo
    .join(inf_costo, on="SKU", how="left")
    .join(skus_campana, on="SKU", how="left")
    .fillna({"es_campana": 0, "COSTO_UNITARIO": 0.0, "VIDA_UTIL": 0.0, "CS_POR_PALLET": 0.0})
)

df_modelo.show(8, truncate=40)
print("SKUs marcados como campaña:", df_modelo.filter(F.col("es_campana") == 1).select("SKU").distinct().count())

+------+----------------------------------------+------------------------------+------+-------+--------------+---------+-------------+----------+
|   SKU|                             DESCRIPCION|                     PROVEEDOR|semana|demanda|COSTO_UNITARIO|VIDA_UTIL|CS_POR_PALLET|es_campana|
+------+----------------------------------------+------------------------------+------+-------+--------------+---------+-------------+----------+
|100396|"BROCHURE HOLDER 3.75X8.5""  (UNIDADES)"|STARBUCKS CORPORATION Utileria|     1|    0.0|          2.74|      0.0|         15.0|         0|
|100396|"BROCHURE HOLDER 3.75X8.5""  (UNIDADES)"|STARBUCKS CORPORATION Utileria|     2|    0.0|          2.74|      0.0|         15.0|         0|
|100396|"BROCHURE HOLDER 3.75X8.5""  (UNIDADES)"|STARBUCKS CORPORATION Utileria|     3|    0.0|          2.74|      0.0|         15.0|         0|
|100396|"BROCHURE HOLDER 3.75X8.5""  (UNIDADES)"|STARBUCKS CORPORATION Utileria|     4|    0.0|          2.74|      0.0|    

SKUs marcados como campaña: 27


## Paso 4 - Persistencia en Parquet

Guardamos las tablas procesadas en formato columnar Parquet (optimizado para Spark) para que los notebooks de modelado las reutilicen sin repetir el preprocesamiento.

In [8]:
# Tabla larga lista para el modelo de regresión.
(df_modelo.write.mode("overwrite").parquet(os.path.join(PARQUET_DIR, "demanda_larga")))

# Tablas de dimensión para el modelo de clustering.
(df_hist.write.mode("overwrite").parquet(os.path.join(PARQUET_DIR, "historico")))
(df_inf.write.mode("overwrite").parquet(os.path.join(PARQUET_DIR, "inf")))
(df_bio.write.mode("overwrite").parquet(os.path.join(PARQUET_DIR, "bio")))

print("Parquet escritos en:", PARQUET_DIR)
print(os.listdir(PARQUET_DIR))

Parquet escritos en: ..\data\parquet
['bio', 'demanda_larga', 'forecast_sku', 'historico', 'inf', 'plan_reabastecimiento', 'segmentos_sku']


In [9]:
spark.stop()
print("SparkSession finalizada.")

SparkSession finalizada.
